# Further Research 4 -- Graduate Underemployment by Cohort

## Research Question
Is underemployment getting worse for each new cohort of Malaysian graduates?

## Background

Malaysia has been producing more graduates every year. University enrolment has
grown steadily, and the share of the population with a degree has risen across
every state. But a degree is only valuable if it leads to work that actually uses
it. When a graduate ends up in a job that does not require their qualification --
serving at a counter, driving a car, doing administrative work that a school
leaver could do -- that is underemployment. The degree was paid for. The sacrifice
was made. But the return never arrived.

The question this research asks is whether this problem is getting better or worse
over time. Specifically: is each new wave of graduates entering the Malaysian
workforce facing a harder situation than the wave that came before them?

This matters because if underemployment is just a short transitional phase --
graduates take a mismatched job for a year or two, then find their footing -- it
is uncomfortable but manageable. The investment in education still pays off,
just with a slight delay. But if underemployment is getting structurally worse
with each cohort, and if it follows graduates into their thirties rather than
resolving in their twenties, then the education system is producing qualifications
that the labour market cannot absorb. Families are paying more for degrees --
as Research 1 and 2 showed -- and getting less in return.

## What Data We Are Using

We have quarterly underemployment data from DOSM from 2017 to 2025, measured
using the Skills-Related Underemployment (SRU) rate. SRU measures the share of
employed graduates who are working in jobs that do not require their level of
education. It is not unemployment -- these people have jobs. The problem is that
the jobs are below what their qualification should get them.

The data is broken down by age group: 15-24, 25-34, 35-44, and 45+. We also have
the same data split by gender.

We cannot follow the same individuals over time. What we can do is use age groups
as proxies for career stage. Think of it like comparing two different runners at
the same starting line but in different years. The 15-24 group in 2017 is one
generation of fresh graduates. The 15-24 group in 2025 is the next generation.
By comparing their starting conditions, we can tell whether the labour market
is getting harder or easier for each new cohort entering it.


## Setup -- Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
RED    = '#DC2626'
GRAY   = '#6B7280'
PURPLE = '#7C3AED'
TEAL   = '#0D9488'
YELLOW = '#D97706'

AGE_COLORS = {
    '15-24': RED,
    '25-34': ORANGE,
    '35-44': BLUE,
    '45+':   GRAY,
    'overall': 'black',
}

print("Libraries loaded.")


In [ ]:
age_raw = pd.read_csv('../data/cleaned/graduate_underemployment_age.csv',
                      parse_dates=['date'])
sex_raw = pd.read_csv('../data/cleaned/graduate_underemployment_sex.csv',
                      parse_dates=['date'])

age_raw['year']    = age_raw['date'].dt.year
age_raw['quarter'] = age_raw['date'].dt.quarter
age_raw['sru_rate'] = pd.to_numeric(age_raw['sru_rate'], errors='coerce')
age_raw['sru_persons'] = pd.to_numeric(age_raw['sru_persons'], errors='coerce')

sex_raw['year']    = sex_raw['date'].dt.year
sex_raw['quarter'] = sex_raw['date'].dt.quarter
sex_raw['sru_rate'] = pd.to_numeric(sex_raw['sru_rate'], errors='coerce')

print("Data loaded:")
print(f"  Age breakdown:    {len(age_raw)} rows")
print(f"  Sex breakdown:    {len(sex_raw)} rows")
print(f"  Date range:       {age_raw['date'].min().date()} to {age_raw['date'].max().date()}")
print(f"  Age groups:       {sorted(age_raw['age'].unique())}")
print(f"  Gender groups:    {sorted(sex_raw['sex'].unique())}")


In [ ]:
# Define COVID period boundaries used throughout the notebook
COVID_START = pd.Timestamp('2020-01-01')
COVID_END   = pd.Timestamp('2021-10-01')

# Define the three comparison periods
PRE_COVID_START  = pd.Timestamp('2017-01-01')
PRE_COVID_END    = pd.Timestamp('2019-10-01')
COVID_PEAK_START = pd.Timestamp('2020-01-01')
COVID_PEAK_END   = pd.Timestamp('2021-10-01')
POST_COVID_START = pd.Timestamp('2022-01-01')
POST_COVID_END   = age_raw['date'].max()

print("Period definitions:")
print(f"  Pre-COVID:  {PRE_COVID_START.date()} to {PRE_COVID_END.date()}")
print(f"  COVID peak: {COVID_PEAK_START.date()} to {COVID_PEAK_END.date()}")
print(f"  Post-COVID: {POST_COVID_START.date()} to {POST_COVID_END.date()}")


In [ ]:
# Extract the overall national rate for easy reference
overall_df = age_raw[age_raw['age'] == 'overall'].sort_values('date').copy()
age_groups = ['15-24', '25-34', '35-44', '45+']
age_dfs    = {ag: age_raw[age_raw['age'] == ag].sort_values('date').copy()
              for ag in age_groups}

print("Quick summary -- average SRU rate by age group across all quarters:")
print(f"{'Age group':<12} {'Mean SRU rate':>15} {'Min':>8} {'Max':>8}")
print("-" * 46)
for ag in age_groups + ['overall']:
    df = age_raw[age_raw['age'] == ag]
    print(f"  {ag:<10} {df['sru_rate'].mean():>13.1f}%  "
          f"{df['sru_rate'].min():>6.1f}%  "
          f"{df['sru_rate'].max():>6.1f}%")


---
# Step 1 -- The Overall Underemployment Picture from 2017 to 2025

Before breaking anything down by age or gender, we look at the national
underemployment rate as a single line across all quarters from 2017 to 2025.

This step answers a simple question first: is underemployment in Malaysia broadly
going up, down, or sideways over this period? The answer sets the context for
everything that follows.

We mark the COVID-19 period as a shaded zone on the chart so the pandemic
distortion is clearly visible. The spike during 2020-2021 is real but it is
a shock, not a trend. What matters most for the research question is what
happened before COVID and what level the rate settled at after COVID passed.


### 1a -- Compute annual averages alongside quarterly data

We use quarterly data for the trend line but also compute annual averages so we
can describe the overall direction in plain numbers. The annual average smooths
out the quarter-to-quarter noise and makes the direction of travel clearer.


In [ ]:
annual_overall = overall_df.groupby('year')['sru_rate'].mean().reset_index()
annual_overall.columns = ['year', 'avg_sru_rate']

print("Annual average overall SRU rate:")
print(f"{'Year':<8} {'Avg SRU rate':>14}")
print("-" * 25)
for _, row in annual_overall.iterrows():
    flag = ' <-- COVID' if row['year'] in [2020, 2021] else ''
    print(f"  {int(row['year']):<6} {row['avg_sru_rate']:>12.1f}%{flag}")


### 1b -- Identify the pre-COVID baseline and post-COVID settlement level

We compute the average SRU rate for the pre-COVID period (2017-2019) and the
post-COVID period (2022-2025). Comparing these two averages directly answers
whether underemployment recovered after the pandemic or settled at a permanently
higher level.


In [ ]:
pre_covid_avg  = overall_df[
    (overall_df['date'] >= PRE_COVID_START) &
    (overall_df['date'] <= PRE_COVID_END)
]['sru_rate'].mean()

post_covid_avg = overall_df[
    overall_df['date'] >= POST_COVID_START
]['sru_rate'].mean()

change = post_covid_avg - pre_covid_avg

print(f"Pre-COVID average SRU rate  (2017-2019): {pre_covid_avg:.1f}%")
print(f"Post-COVID average SRU rate (2022-2025): {post_covid_avg:.1f}%")
print(f"Change:                                  {change:+.1f} percentage points")
print()
if change > 0:
    print("The post-COVID level is HIGHER than pre-COVID.")
    print("Underemployment did not fully recover after the pandemic.")
elif change < 0:
    print("The post-COVID level is LOWER than pre-COVID.")
    print("Underemployment actually improved relative to before the pandemic.")
else:
    print("The post-COVID level is the same as pre-COVID.")


### 1c -- Fit a trend line to the pre-COVID and post-COVID periods separately

A single trend line across the full period would be misleading because COVID
creates a huge spike in the middle. Instead we fit two separate trend lines --
one for 2017-2019 and one for 2022-2025 -- to see the direction of travel
before and after the pandemic independently.


In [ ]:
pre_df  = overall_df[
    (overall_df['date'] >= PRE_COVID_START) &
    (overall_df['date'] <= PRE_COVID_END)
].copy()
post_df = overall_df[overall_df['date'] >= POST_COVID_START].copy()

pre_df['t']  = (pre_df['date']  - pre_df['date'].min()).dt.days
post_df['t'] = (post_df['date'] - post_df['date'].min()).dt.days

pre_slope,  pre_intercept,  *_ = stats.linregress(pre_df['t'],  pre_df['sru_rate'])
post_slope, post_intercept, *_ = stats.linregress(post_df['t'], post_df['sru_rate'])

pre_trend_annualised  = pre_slope  * 365
post_trend_annualised = post_slope * 365

print("Trend line slopes (annualised):")
print(f"  Pre-COVID  (2017-2019): {pre_trend_annualised:+.2f} pp/yr")
print(f"  Post-COVID (2022-2025): {post_trend_annualised:+.2f} pp/yr")
print()
if post_trend_annualised > 0:
    print("Post-COVID trend is RISING -- underemployment is getting worse after recovery.")
elif post_trend_annualised < 0:
    print("Post-COVID trend is FALLING -- underemployment is improving after recovery.")


### 1d -- Plot the overall SRU rate with COVID zone and trend lines

The quarterly rate as a line, the COVID period shaded in light red, the pre-COVID
average as a dotted horizontal line for reference, and the two trend lines overlaid.
This single chart tells the full national story.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# COVID shading
ax.axvspan(COVID_START, COVID_END, alpha=0.12, color=RED, label='COVID-19 period')

# Quarterly rate
ax.plot(overall_df['date'], overall_df['sru_rate'],
        color='black', lw=2, marker='o', markersize=4, label='Overall SRU rate (quarterly)')

# Pre-COVID average reference line
ax.axhline(pre_covid_avg, color=BLUE, lw=1.5, ls='--', alpha=0.7,
           label=f'Pre-COVID avg ({pre_covid_avg:.1f}%)')

# Pre-COVID trend line
pre_x = [pre_df['date'].min(), pre_df['date'].max()]
pre_y = [pre_intercept, pre_intercept + pre_slope * pre_df['t'].max()]
ax.plot(pre_x, pre_y, color=GREEN, lw=2, ls='-', alpha=0.8,
        label=f'Pre-COVID trend ({pre_trend_annualised:+.2f} pp/yr)')

# Post-COVID trend line
post_x = [post_df['date'].min(), post_df['date'].max()]
post_y = [post_intercept, post_intercept + post_slope * post_df['t'].max()]
ax.plot(post_x, post_y, color=ORANGE, lw=2, ls='-', alpha=0.8,
        label=f'Post-COVID trend ({post_trend_annualised:+.2f} pp/yr)')

ax.set_xlabel('Quarter')
ax.set_ylabel('SRU Rate (%)')
ax.set_title('Step 1 -- Overall Graduate Underemployment Rate (2017-2025)\n'
             'SRU = Skills-Related Underemployment | Employed graduates in below-qualification jobs',
             fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Overall SRU rate ranged from {overall_df['sru_rate'].min():.1f}% "
      f"to {overall_df['sru_rate'].max():.1f}% across all quarters.")


---
# Step 2 -- How Does Each Age Group Compare?

Now we split by age group. Each age group represents a different career stage.
The 15-24 group are fresh graduates -- they just entered the workforce and have
not yet had time to find roles that match their qualification. The 25-34 group
entered the workforce roughly a decade earlier. The 35-44 and 45+ groups are
mid and late career.

What we are watching for is not just how high each line sits, but whether the
gaps between lines are widening or narrowing over time. If the 15-24 line is
not falling as fast as the older groups, it means each new wave of graduates
is entering a harder market than the cohort ahead of them -- even if the overall
national rate looks like it is improving.


### 2a -- Compute annual averages for each age group

We compute the annual average SRU rate for each age group so we can see the
long-run direction clearly without quarterly noise. We also compute the gap
between each age group and the 45+ group, which represents the most settled
workers in the labour market.


In [ ]:
annual_by_age = age_raw[age_raw['age'] != 'overall'].groupby(
    ['year', 'age'])['sru_rate'].mean().reset_index()
annual_by_age.columns = ['year', 'age', 'avg_sru_rate']

print("Annual average SRU rate by age group:")
print(f"{'Year':<8}", end='')
for ag in age_groups:
    print(f"  {ag:>8}", end='')
print()
print("-" * 50)
for yr in sorted(annual_by_age['year'].unique()):
    yr_data = annual_by_age[annual_by_age['year']==yr]
    print(f"  {yr:<6}", end='')
    for ag in age_groups:
        val = yr_data[yr_data['age']==ag]['avg_sru_rate'].values
        print(f"  {val[0]:>6.1f}%" if len(val)>0 else f"  {'n/a':>6}", end='')
    print()


### 2b -- Compute the gap between the 15-24 group and the 25-34 group each year

The gap between these two groups is particularly meaningful. It tells us how much
worse off fresh graduates are compared to graduates who entered the workforce just
one decade earlier. A widening gap means the starting position for new graduates
is deteriorating relative to those who came before them.


In [ ]:
gap_rows = []
for yr in sorted(annual_by_age['year'].unique()):
    yr_data  = annual_by_age[annual_by_age['year']==yr]
    rate_1524 = yr_data[yr_data['age']=='15-24']['avg_sru_rate'].values
    rate_2534 = yr_data[yr_data['age']=='25-34']['avg_sru_rate'].values
    if len(rate_1524) > 0 and len(rate_2534) > 0:
        gap_rows.append({
            'year':      yr,
            'rate_1524': rate_1524[0],
            'rate_2534': rate_2534[0],
            'gap':       rate_1524[0] - rate_2534[0],
        })

gap_df = pd.DataFrame(gap_rows)

print("Gap between 15-24 and 25-34 SRU rate (annual average):")
print(f"{'Year':<8} {'15-24 rate':>12} {'25-34 rate':>12} {'Gap':>10}")
print("-" * 46)
for _, row in gap_df.iterrows():
    flag = ' <-- COVID' if row['year'] in [2020, 2021] else ''
    print(f"  {int(row['year']):<6} {row['rate_1524']:>10.1f}%  "
          f"{row['rate_2534']:>10.1f}%  "
          f"{row['gap']:>8.1f}pp{flag}")


### 2c -- Plot all four age groups on the same chart

Four lines on the same chart, each a different colour. The COVID shading is
included again. We are looking for whether the lines are converging -- meaning
younger graduates are catching up to older cohorts -- or diverging -- meaning
the gap between young and old is growing.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)
ax.text(pd.Timestamp('2020-04-01'), ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 70,
        'COVID-19', fontsize=8, color=RED, alpha=0.7)

for ag in age_groups:
    df = age_dfs[ag]
    ax.plot(df['date'], df['sru_rate'],
            color=AGE_COLORS[ag], lw=2.5, marker='o', markersize=4,
            label=f'Age {ag}')

ax.set_xlabel('Quarter')
ax.set_ylabel('SRU Rate (%)')
ax.set_title('Step 2 -- Graduate Underemployment Rate by Age Group (2017-2025)\n'
             'Are the gaps between age groups widening or narrowing?',
             fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### 2d -- Plot the gap between 15-24 and 25-34 over time

A single bar chart showing the gap between the youngest and the next age group
each year. If the bars are growing taller over time (excluding COVID), it means
fresh graduates are falling further behind their slightly older peers each year.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

non_covid = gap_df[~gap_df['year'].isin([2020, 2021])]
covid     = gap_df[gap_df['year'].isin([2020, 2021])]

ax.bar(non_covid['year'], non_covid['gap'], color=RED,   alpha=0.85,
       label='Normal period')
ax.bar(covid['year'],     covid['gap'],     color=GRAY,  alpha=0.60,
       label='COVID period (excluded from trend)')

slope_gap, intercept_gap, *_ = stats.linregress(
    non_covid['year'], non_covid['gap'])
trend_x = non_covid['year'].values
trend_y = intercept_gap + slope_gap * trend_x
ax.plot(trend_x, trend_y, color='black', lw=2, ls='--',
        label=f'Trend ({slope_gap:+.2f} pp/yr)')

ax.set_xlabel('Year')
ax.set_ylabel('Gap (pp)')
ax.set_title('Gap Between 15-24 and 25-34 SRU Rate\n'
             'Growing gap = fresh graduates falling further behind',
             fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Trend in gap (excluding COVID): {slope_gap:+.2f} pp/yr")
if slope_gap > 0:
    print("The gap is WIDENING -- fresh graduates are falling further behind each year.")
else:
    print("The gap is NARROWING -- fresh graduates are catching up to older cohorts.")


---
# Step 3 -- Is the Youngest Cohort Getting a Worse Start Than Before?

This step focuses only on the 15-24 age group and compares their situation
across three periods: before COVID, during COVID, and after COVID.

The key question is what happened after COVID. Did the 15-24 underemployment
rate recover to where it was before the pandemic, or did it settle at a
permanently higher level? If it did not recover, then every fresh graduate
entering the workforce from 2022 onwards is starting from a worse position
than graduates who entered in 2017-2019 -- even though the pandemic is over.


### 3a -- Compute the average 15-24 SRU rate for each of the three periods

We compute the average SRU rate for the 15-24 group within each period.
The pre-COVID average is the benchmark. We compare the COVID peak and the
post-COVID settlement level against it.


In [ ]:
df_1524 = age_dfs['15-24']

pre_1524  = df_1524[
    (df_1524['date'] >= PRE_COVID_START) &
    (df_1524['date'] <= PRE_COVID_END)
]['sru_rate'].mean()

covid_1524 = df_1524[
    (df_1524['date'] >= COVID_PEAK_START) &
    (df_1524['date'] <= COVID_PEAK_END)
]['sru_rate'].mean()

post_1524 = df_1524[
    df_1524['date'] >= POST_COVID_START
]['sru_rate'].mean()

recovery  = post_1524 - pre_1524

print("15-24 age group SRU rate by period:")
print(f"  Pre-COVID  (2017-2019): {pre_1524:.1f}%")
print(f"  COVID peak (2020-2021): {covid_1524:.1f}%")
print(f"  Post-COVID (2022-2025): {post_1524:.1f}%")
print()
print(f"  Change from pre-COVID to post-COVID: {recovery:+.1f} pp")
print()
if recovery > 1:
    print("  The 15-24 rate is HIGHER post-COVID than pre-COVID.")
    print("  Fresh graduates today face a harder starting position than in 2017-2019.")
elif recovery < -1:
    print("  The 15-24 rate is LOWER post-COVID than pre-COVID.")
    print("  Fresh graduates today face an easier starting position than in 2017-2019.")
else:
    print("  The 15-24 rate is roughly the same post-COVID as pre-COVID.")
    print("  The pandemic caused a spike but the rate returned to its original level.")


### 3b -- Fit a trend line within the post-COVID period for the 15-24 group

Even if the post-COVID level is higher than pre-COVID, the direction matters.
Is it still falling -- meaning it will eventually return to the pre-COVID level?
Or is it flat or rising -- meaning the new, higher level is becoming permanent?


In [ ]:
post_1524_df = df_1524[df_1524['date'] >= POST_COVID_START].copy()
post_1524_df['t'] = (post_1524_df['date'] - post_1524_df['date'].min()).dt.days

slope_post_1524, intercept_post_1524, *_ = stats.linregress(
    post_1524_df['t'], post_1524_df['sru_rate'])
trend_annualised_1524 = slope_post_1524 * 365

print(f"Post-COVID trend for 15-24 group: {trend_annualised_1524:+.2f} pp/yr")
print()
if trend_annualised_1524 > 0.5:
    print("The trend is RISING within the post-COVID period.")
    print("The starting position for fresh graduates is getting worse, not better.")
elif trend_annualised_1524 < -0.5:
    yrs_to_recover = abs(recovery / trend_annualised_1524)
    print(f"The trend is FALLING at {abs(trend_annualised_1524):.2f} pp/yr.")
    print(f"At this rate it would take approximately {yrs_to_recover:.0f} more years")
    print(f"to return to the pre-COVID average of {pre_1524:.1f}%.")
else:
    print("The trend is roughly FLAT within the post-COVID period.")
    print("The higher post-COVID level appears to be the new normal.")


### 3c -- Plot the 15-24 SRU rate with period averages and post-COVID trend

The quarterly 15-24 rate as a line. Three horizontal lines showing the average
for each period. The post-COVID trend overlaid so the direction of travel is visible.
The pre-COVID average is the reference point the reader's eye returns to.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.axvspan(COVID_START, COVID_END, alpha=0.12, color=RED)

# Quarterly 15-24 rate
ax.plot(df_1524['date'], df_1524['sru_rate'],
        color=RED, lw=2.5, marker='o', markersize=5,
        label='15-24 SRU rate (quarterly)')

# Period average lines
ax.axhline(pre_1524,  color=GREEN,  lw=2, ls='--', alpha=0.8,
           label=f'Pre-COVID avg ({pre_1524:.1f}%)')
ax.axhline(covid_1524, color=GRAY,   lw=1.5, ls=':', alpha=0.6,
           label=f'COVID avg ({covid_1524:.1f}%)')
ax.axhline(post_1524, color=ORANGE, lw=2, ls='--', alpha=0.8,
           label=f'Post-COVID avg ({post_1524:.1f}%)')

# Post-COVID trend line
post_x_plot = [post_1524_df['date'].min(), post_1524_df['date'].max()]
post_y_plot = [intercept_post_1524,
               intercept_post_1524 + slope_post_1524 * post_1524_df['t'].max()]
ax.plot(post_x_plot, post_y_plot, color='black', lw=2, ls='-',
        label=f'Post-COVID trend ({trend_annualised_1524:+.2f} pp/yr)')

# Annotate the gap between pre and post COVID averages
ax.annotate('', xy=(pd.Timestamp('2025-01-01'), post_1524),
            xytext=(pd.Timestamp('2025-01-01'), pre_1524),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(pd.Timestamp('2025-03-01'), (pre_1524 + post_1524)/2,
        f'{recovery:+.1f}pp', fontsize=9, va='center', fontweight='bold')

ax.set_xlabel('Quarter')
ax.set_ylabel('SRU Rate (%)')
ax.set_title('Step 3 -- Did the 15-24 Underemployment Rate Recover After COVID?\n'
             'Pre-COVID average is the benchmark for each new cohort of graduates',
             fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 3d -- Plain language summary of Step 3

We translate the numbers into a direct answer to the research question
for this step.


In [ ]:
print("STEP 3 SUMMARY")
print("=" * 55)
print()
print(f"Before COVID (2017-2019), fresh graduates (15-24) had")
print(f"an average underemployment rate of {pre_1524:.1f}%.")
print()
print(f"After COVID (2022-2025), the rate settled at {post_1524:.1f}%.")
print()
if abs(recovery) < 1:
    print("This is roughly the same -- the pandemic was a temporary shock")
    print("and the rate returned to its pre-pandemic level.")
elif recovery > 0:
    print(f"This is {recovery:.1f} percentage points HIGHER than before.")
    print("Fresh graduates entering the workforce from 2022 onwards")
    print("face a harder starting position than those who entered in 2017.")
    print()
    print("In plain terms: out of every 100 fresh graduates, roughly")
    print(f"{recovery:.0f} more are underemployed today than were in 2017-2019.")
else:
    print(f"This is {abs(recovery):.1f} percentage points LOWER than before.")
    print("The labour market for fresh graduates has actually improved")
    print("relative to the pre-pandemic baseline.")


---
# Step 4 -- Does Underemployment Follow Graduates as They Age?

This step uses the age groups as a tracking mechanism across time. The graduates
who were 15-24 in 2017 are now roughly 23-32 in 2025 -- which means they fall
in the 25-34 age bucket today.

We cannot follow the exact same individuals, but we can compare what the
25-34 rate looks like in 2025 to what the 15-24 rate looked like in 2017
for the same approximate group of people. If the 25-34 rate in 2025 is still
high, it means these graduates did not escape underemployment as they aged --
it followed them through their careers.

This is the difference between underemployment as a temporary transition problem
and underemployment as a permanent structural one.


### 4a -- Extract the cohort tracking data points

We identify the specific data points needed to trace each cohort. The 2017
cohort was 15-24 in 2017 and is 25-34 in 2025. The 2019 cohort was 15-24
in 2019 and is roughly 25-34 in 2027, which is outside our data window, so
we use 2024 as a proxy. We extract these specific rate values.


In [ ]:
# Annual averages for the cohort trace
annual_1524 = age_raw[age_raw['age']=='15-24'].groupby('year')['sru_rate'].mean()
annual_2534 = age_raw[age_raw['age']=='25-34'].groupby('year')['sru_rate'].mean()

# Cohort traces: age 15-24 in year X maps to age 25-34 roughly 10 years later
# With data from 2017-2025 we can do:
# Cohort A: 15-24 in 2017 -> 25-34 in 2025 (8 year gap, approximate)
# Cohort B: 15-24 in 2019 -> 25-34 in 2024 (5 year gap, shorter but usable)

cohort_traces = []
for start_yr in [2017, 2018, 2019]:
    end_yr = start_yr + 7
    if end_yr in annual_2534.index and start_yr in annual_1524.index:
        cohort_traces.append({
            'cohort_label': f'Entered ~{start_yr}',
            'start_year':   start_yr,
            'end_year':     end_yr,
            'rate_young':   annual_1524[start_yr],
            'rate_older':   annual_2534[end_yr],
            'change':       annual_2534[end_yr] - annual_1524[start_yr],
        })

print("Cohort traces -- SRU rate when young vs SRU rate 7 years later:")
print(f"{'Cohort':<18} {'15-24 rate (entry)':>20} {'25-34 rate (7yr later)':>24} {'Change':>10}")
print("-" * 76)
for ct in cohort_traces:
    print(f"  {ct['cohort_label']:<16} "
          f"{ct['rate_young']:>18.1f}%  "
          f"{ct['rate_older']:>22.1f}%  "
          f"{ct['change']:>+8.1f}pp")
print()
print("A large negative change means the cohort improved significantly as they aged.")
print("A small or positive change means underemployment persisted into their 30s.")


### 4b -- Plot the full age group trajectories with cohort traces overlaid

We plot all four age group lines from 2017 to 2025. On top of that we draw
diagonal arrows connecting the cohort starting point (15-24 rate in year X)
to the cohort landing point (25-34 rate in year X+7). The length of the drop
along the arrow tells you how much the cohort improved as they aged.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)

for ag in age_groups:
    df = age_dfs[ag]
    ax.plot(df['date'], df['sru_rate'],
            color=AGE_COLORS[ag], lw=2, marker='o', markersize=3,
            alpha=0.7, label=f'Age {ag}')

# Cohort trace arrows
arrow_colors = ['#7C3AED', '#0891B2', '#B45309']
for ct, col in zip(cohort_traces, arrow_colors):
    x_start = pd.Timestamp(f"{ct['start_year']}-06-01")
    x_end   = pd.Timestamp(f"{ct['end_year']}-06-01")
    y_start = ct['rate_young']
    y_end   = ct['rate_older']
    ax.annotate('',
        xy=(x_end, y_end), xytext=(x_start, y_start),
        arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.scatter([x_start, x_end], [y_start, y_end],
               color=col, s=80, zorder=5)
    mid_x = x_start + (x_end - x_start) / 2
    mid_y = (y_start + y_end) / 2
    ax.text(mid_x, mid_y + 1.5, ct['cohort_label'],
            fontsize=8, color=col, ha='center', fontweight='bold')

ax.set_xlabel('Year')
ax.set_ylabel('SRU Rate (%)')
ax.set_title('Step 4 -- Does Underemployment Follow Graduates as They Age?\n'
             'Arrows trace each cohort from entry (15-24) to 7 years later (25-34)',
             fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 4c -- Quantify how much the 25-34 rate has changed over the data window

If underemployment is a temporary entry problem, we would expect the 25-34
rate to be falling as each cohort settles into better-matched roles. If the
25-34 rate is flat or rising, it means the problem is structural -- graduates
are staying underemployed even into their thirties.


In [ ]:
pre_2534  = age_dfs['25-34'][
    (age_dfs['25-34']['date'] >= PRE_COVID_START) &
    (age_dfs['25-34']['date'] <= PRE_COVID_END)
]['sru_rate'].mean()

post_2534 = age_dfs['25-34'][
    age_dfs['25-34']['date'] >= POST_COVID_START
]['sru_rate'].mean()

change_2534 = post_2534 - pre_2534

print("25-34 age group SRU rate -- pre vs post COVID:")
print(f"  Pre-COVID  (2017-2019): {pre_2534:.1f}%")
print(f"  Post-COVID (2022-2025): {post_2534:.1f}%")
print(f"  Change:                 {change_2534:+.1f} pp")
print()
print("Comparison -- did the 15-24 group or 25-34 group change more?")
print(f"  15-24 change: {recovery:+.1f} pp")
print(f"  25-34 change: {change_2534:+.1f} pp")
print()
if abs(recovery) > abs(change_2534):
    print("The 15-24 group changed more -- fresh graduates are bearing")
    print("a disproportionate share of the worsening underemployment.")
else:
    print("The 25-34 group changed more -- the problem is spreading")
    print("into the mid-career stage, not just affecting new entrants.")


### 4d -- Plot the 15-24 and 25-34 rates side by side for direct comparison

Two lines only -- the two age groups that represent the entry and early career
stages. Placing them together on a single chart with the pre-COVID averages
marked for both makes the structural picture immediately clear.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)

ax.plot(age_dfs['15-24']['date'], age_dfs['15-24']['sru_rate'],
        color=RED,    lw=2.5, marker='o', markersize=4, label='Age 15-24 (fresh graduates)')
ax.plot(age_dfs['25-34']['date'], age_dfs['25-34']['sru_rate'],
        color=ORANGE, lw=2.5, marker='s', markersize=4, label='Age 25-34 (early career)')

ax.axhline(pre_1524,  color=RED,    lw=1.2, ls=':', alpha=0.6,
           label=f'15-24 pre-COVID avg ({pre_1524:.1f}%)')
ax.axhline(pre_2534,  color=ORANGE, lw=1.2, ls=':', alpha=0.6,
           label=f'25-34 pre-COVID avg ({pre_2534:.1f}%)')

ax.set_xlabel('Quarter')
ax.set_ylabel('SRU Rate (%)')
ax.set_title('Step 4 -- Fresh Graduates (15-24) vs Early Career (25-34)\n'
             'Dotted lines = pre-COVID averages (the benchmark for each group)',
             fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


---
# Step 5 -- Is the Burden Falling Equally on Men and Women?

We now use the gender breakdown of the SRU data. If women are consistently more
underemployed than men despite similar or higher educational attainment, the
return on their education investment is structurally lower.

This connects directly to the broader EduNilai story. Research 1 and 2 showed
that the financial sacrifice to access a degree is already high and unevenly
distributed by geography. If women then face systematically worse employment
outcomes from the same degree, they are doubly penalised -- paying more in
relative terms and getting less in return.


### 5a -- Compute the male-female SRU rate and their gap over time

We extract the male and female rates for each quarter and compute the gap
between them. A positive gap means women are more underemployed. We also
compute the annual average gap so the trend is easier to read.


In [ ]:
female_df = sex_raw[sex_raw['sex']=='female'].sort_values('date').copy()
male_df   = sex_raw[sex_raw['sex']=='male'].sort_values('date').copy()

gender_gap = female_df[['date','year','sru_rate']].merge(
    male_df[['date','sru_rate']], on='date', suffixes=('_female','_male'))
gender_gap['gap'] = gender_gap['sru_rate_female'] - gender_gap['sru_rate_male']

pre_gap  = gender_gap[gender_gap['date'] <= PRE_COVID_END]['gap'].mean()
post_gap = gender_gap[gender_gap['date'] >= POST_COVID_START]['gap'].mean()

print("Gender gap in SRU rate (female minus male):")
print(f"{'Quarter':<14} {'Female':>10} {'Male':>10} {'Gap':>8}")
print("-" * 46)
for _, row in gender_gap.iterrows():
    print(f"  {str(row['date'].date()):<12} "
          f"{row['sru_rate_female']:>8.1f}%  "
          f"{row['sru_rate_male']:>8.1f}%  "
          f"{row['gap']:>+6.1f}pp")
print()
print(f"Average gap pre-COVID  (2017-2019): {pre_gap:+.1f} pp")
print(f"Average gap post-COVID (2022-2025): {post_gap:+.1f} pp")


### 5b -- Is the gender gap widening or narrowing?

We fit a trend line on the gender gap (excluding COVID) to see whether the
difference between male and female underemployment is growing or shrinking
over time.


In [ ]:
non_covid_gap = gender_gap[
    ~gender_gap['date'].between(COVID_PEAK_START, COVID_PEAK_END)
].copy()
non_covid_gap['t'] = (non_covid_gap['date'] - non_covid_gap['date'].min()).dt.days

slope_gap_gender, intercept_gap_gender, *_ = stats.linregress(
    non_covid_gap['t'], non_covid_gap['gap'])
trend_gap_annual = slope_gap_gender * 365

print(f"Gender gap trend (excluding COVID): {trend_gap_annual:+.2f} pp/yr")
print()
if trend_gap_annual > 0.2:
    print("The gap is WIDENING -- women are falling further behind men each year.")
elif trend_gap_annual < -0.2:
    print("The gap is NARROWING -- the gender disadvantage is reducing over time.")
else:
    print("The gap is roughly STABLE -- the gender disadvantage is persistent but not worsening.")


### 5c -- Plot male and female SRU rates on the same chart

Two lines with the gender gap shaded between them. A wider shaded area means
a larger gap between male and female underemployment. We mark the pre-COVID
period averages for both as reference lines.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: male and female rates with shaded gap
axes[0].axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)
axes[0].plot(female_df['date'], female_df['sru_rate'],
             color=ORANGE, lw=2.5, marker='o', markersize=4, label='Female')
axes[0].plot(male_df['date'],   male_df['sru_rate'],
             color=BLUE,   lw=2.5, marker='s', markersize=4, label='Male')
axes[0].fill_between(female_df['date'],
                     female_df['sru_rate'].values,
                     male_df['sru_rate'].values,
                     alpha=0.15, color=ORANGE, label='Gender gap')
pre_female = female_df[female_df['date'] <= PRE_COVID_END]['sru_rate'].mean()
pre_male   = male_df[male_df['date']     <= PRE_COVID_END]['sru_rate'].mean()
axes[0].axhline(pre_female, color=ORANGE, lw=1.2, ls=':', alpha=0.6,
                label=f'Female pre-COVID avg ({pre_female:.1f}%)')
axes[0].axhline(pre_male,   color=BLUE,   lw=1.2, ls=':', alpha=0.6,
                label=f'Male pre-COVID avg ({pre_male:.1f}%)')
axes[0].set_xlabel('Quarter')
axes[0].set_ylabel('SRU Rate (%)')
axes[0].set_title('Male vs Female SRU Rate\n(Shaded = gender gap)', fontweight='bold')
axes[0].legend(fontsize=8)

# Right: gender gap over time with trend
axes[1].axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)
bar_colors_g = [GRAY if d >= COVID_PEAK_START and d <= COVID_PEAK_END
                else ORANGE for d in gender_gap['date']]
axes[1].bar(gender_gap['date'], gender_gap['gap'],
            width=60, color=bar_colors_g, alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
trend_x_g = [non_covid_gap['date'].min(), non_covid_gap['date'].max()]
trend_y_g  = [intercept_gap_gender,
              intercept_gap_gender + slope_gap_gender * non_covid_gap['t'].max()]
axes[1].plot(trend_x_g, trend_y_g, color='black', lw=2, ls='--',
             label=f'Trend ({trend_gap_annual:+.2f} pp/yr)')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Gender Gap (pp) -- Female minus Male')
axes[1].set_title('Gender Gap in SRU Rate Over Time\n'
                  'Positive = women more underemployed | Gray = COVID period',
                  fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Step 5 -- Gender Breakdown of Graduate Underemployment',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


### 5d -- Plain language summary of the gender finding


In [ ]:
print("STEP 5 SUMMARY")
print("=" * 55)
print()
post_female = female_df[female_df['date'] >= POST_COVID_START]['sru_rate'].mean()
post_male   = male_df[male_df['date']     >= POST_COVID_START]['sru_rate'].mean()
print(f"Post-COVID average SRU rate:")
print(f"  Women: {post_female:.1f}%")
print(f"  Men:   {post_male:.1f}%")
print(f"  Gap:   {post_female - post_male:+.1f} pp (women minus men)")
print()
if post_female > post_male:
    print(f"Women are {post_female - post_male:.1f} percentage points more likely")
    print("to be underemployed than men after completing the same level of education.")
    print()
    print("This means the return on education investment is structurally lower")
    print("for women -- they pay the same cost for a degree but face worse")
    print("employment outcomes on average.")
else:
    print("Men are more underemployed than women in the post-COVID period.")
print()
print(f"Gender gap trend: {trend_gap_annual:+.2f} pp/yr (excluding COVID)")


---
# Step 6 -- Summary Dashboard and Final Answer

This step brings together the key findings from all five steps into a single
dashboard and answers the research question directly.

The three charts show:
1. Whether the 15-24 rate recovered after COVID or settled at a permanently higher level
2. Whether the gap between fresh graduates and early career workers is widening
3. Whether the gender gap in underemployment is growing or stable

Together they give a direct yes or no answer to the question: is underemployment
getting worse for each new cohort of Malaysian graduates?


### 6a -- Print the complete summary table before plotting


In [ ]:
print("=" * 65)
print("COMPLETE SUMMARY -- GRADUATE UNDEREMPLOYMENT BY COHORT")
print("=" * 65)
print()
print("OVERALL NATIONAL RATE:")
print(f"  Pre-COVID avg:  {pre_covid_avg:.1f}%")
print(f"  Post-COVID avg: {post_covid_avg:.1f}%")
print(f"  Change:         {post_covid_avg - pre_covid_avg:+.1f} pp")
print(f"  Post-COVID trend: {post_trend_annualised:+.2f} pp/yr")
print()
print("FRESH GRADUATES (15-24):")
print(f"  Pre-COVID avg:  {pre_1524:.1f}%")
print(f"  Post-COVID avg: {post_1524:.1f}%")
print(f"  Change:         {recovery:+.1f} pp")
print(f"  Post-COVID trend: {trend_annualised_1524:+.2f} pp/yr")
print()
print("EARLY CAREER (25-34):")
print(f"  Pre-COVID avg:  {pre_2534:.1f}%")
print(f"  Post-COVID avg: {post_2534:.1f}%")
print(f"  Change:         {change_2534:+.1f} pp")
print()
print("AGE GAP (15-24 minus 25-34):")
gap_pre  = pre_1524  - pre_2534
gap_post = post_1524 - post_2534
print(f"  Pre-COVID avg gap:  {gap_pre:.1f} pp")
print(f"  Post-COVID avg gap: {gap_post:.1f} pp")
print(f"  Gap trend (excl. COVID): {slope_gap:+.2f} pp/yr")
print()
print("GENDER GAP (female minus male):")
print(f"  Pre-COVID avg gap:  {pre_gap:+.1f} pp")
print(f"  Post-COVID avg gap: {post_gap:+.1f} pp")
print(f"  Gap trend (excl. COVID): {trend_gap_annual:+.2f} pp/yr")


### 6b -- Three-chart summary dashboard


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# Chart 1: 15-24 rate with pre and post COVID averages
axes[0].axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)
axes[0].plot(df_1524['date'], df_1524['sru_rate'],
             color=RED, lw=2, marker='o', markersize=3, label='15-24 rate')
axes[0].axhline(pre_1524,  color=GREEN,  lw=2, ls='--',
                label=f'Pre-COVID ({pre_1524:.1f}%)')
axes[0].axhline(post_1524, color=ORANGE, lw=2, ls='--',
                label=f'Post-COVID ({post_1524:.1f}%)')
axes[0].set_title('Did Fresh Graduate Underemployment\nRecover After COVID?',
                  fontweight='bold')
axes[0].set_ylabel('SRU Rate (%)')
axes[0].legend(fontsize=8)

# Chart 2: gap between 15-24 and 25-34
non_covid_g  = gap_df[~gap_df['year'].isin([2020, 2021])]
covid_g      = gap_df[gap_df['year'].isin([2020, 2021])]
axes[1].bar(non_covid_g['year'], non_covid_g['gap'],
            color=RED,  alpha=0.85, label='Normal period')
axes[1].bar(covid_g['year'],     covid_g['gap'],
            color=GRAY, alpha=0.60, label='COVID (excluded)')
axes[1].plot(non_covid_g['year'],
             intercept_gap + slope_gap * non_covid_g['year'],
             color='black', lw=2, ls='--',
             label=f'Trend ({slope_gap:+.2f} pp/yr)')
axes[1].set_title('Gap Between 15-24 and 25-34\n'
                  'Growing = fresh graduates falling further behind',
                  fontweight='bold')
axes[1].set_ylabel('Gap (pp)')
axes[1].legend(fontsize=8)

# Chart 3: gender gap over time
axes[2].axvspan(COVID_START, COVID_END, alpha=0.10, color=RED)
axes[2].bar(gender_gap['date'], gender_gap['gap'],
            width=60, color=bar_colors_g, alpha=0.85, label='Gender gap')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].plot(trend_x_g, trend_y_g, color='black', lw=2, ls='--',
             label=f'Trend ({trend_gap_annual:+.2f} pp/yr)')
axes[2].set_title('Gender Gap in Underemployment\n'
                  'Positive = women more underemployed',
                  fontweight='bold')
axes[2].set_ylabel('Gap (pp) -- Female minus Male')
axes[2].legend(fontsize=8)

plt.suptitle('Step 6 -- Summary: Is Underemployment Getting Worse for New Cohorts?',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


### 6c -- Final answer to the research question


In [ ]:
print("=" * 65)
print("FINAL ANSWER")
print("=" * 65)
print()
print("Research question: Is underemployment getting worse for each")
print("new cohort of Malaysian graduates?")
print()

worsening_signals = 0
total_signals     = 3

if recovery > 1:
    worsening_signals += 1
    print("Signal 1 -- WORSENING:")
    print(f"  The 15-24 SRU rate is {recovery:.1f} pp higher post-COVID than pre-COVID.")
    print("  Fresh graduates today face a harder starting position than in 2017-2019.")
else:
    print("Signal 1 -- NOT WORSENING:")
    print(f"  The 15-24 SRU rate recovered to near its pre-COVID level.")

print()
if slope_gap > 0.3:
    worsening_signals += 1
    print("Signal 2 -- WORSENING:")
    print(f"  The gap between 15-24 and 25-34 is widening at {slope_gap:.2f} pp/yr.")
    print("  Fresh graduates are falling further behind their slightly older peers.")
else:
    print("Signal 2 -- NOT WORSENING:")
    print(f"  The age gap is stable or narrowing ({slope_gap:+.2f} pp/yr).")

print()
if trend_gap_annual > 0.2:
    worsening_signals += 1
    print("Signal 3 -- WORSENING:")
    print(f"  The gender gap is widening at {trend_gap_annual:.2f} pp/yr.")
    print("  Women are bearing a growing share of the underemployment burden.")
else:
    print("Signal 3 -- NOT WORSENING:")
    print(f"  The gender gap is stable or narrowing ({trend_gap_annual:+.2f} pp/yr).")

print()
print(f"Overall: {worsening_signals} out of {total_signals} signals point to worsening.")
print()
if worsening_signals >= 2:
    print("CONCLUSION: YES -- underemployment is getting worse for new cohorts.")
    print("The data shows that graduates entering the Malaysian workforce today")
    print("face a harder labour market than those who entered before them.")
    print("This is not a temporary post-pandemic issue -- it is a structural trend.")
elif worsening_signals == 1:
    print("CONCLUSION: MIXED -- some signals point to worsening, others do not.")
    print("The picture is not clear-cut. Some cohort groups are worse off")
    print("but the overall trend has not definitively worsened.")
else:
    print("CONCLUSION: NO -- the data does not show systematic worsening.")
    print("Underemployment remains high but is not getting worse for new cohorts")
    print("compared to those who came before them.")
